# Refined ASCENTRA Academic Risk Dataset


In [2]:
import numpy as np
import pandas as pd

SOURCE_DATA_PATH = "../data/ascentra_academic_risk.csv"
REFINED_DATA_PATH = "../data/ascentra_student_data_refined.csv"
RANDOM_SEED = 42

rng = np.random.default_rng(RANDOM_SEED)
old_df = pd.read_csv(SOURCE_DATA_PATH)
target_column = "academic_risk" if "academic_risk" in old_df.columns else old_df.columns[-1]

print("Source shape:", old_df.shape)
print("Detected target column:", target_column)


Source shape: (4424, 16)
Detected target column: academic_risk


In [3]:
removed_columns = [
    "quiz_average",
    "late_submission_count",
    "lms_resource_access",
    "assignment_average",
    "lms_login_frequency",
    "ct1_score",
    "ct2_score",
]

base_academic_columns = [
    "student_id",
    "Curricular units 1st sem (approved)",
    "Curricular units 1st sem (grade)",
    "Curricular units 1st sem (enrolled)",
    "Curricular units 1st sem (evaluations)",
    "Admission grade",
]

available_base_columns = [
    col for col in base_academic_columns
    if col in old_df.columns
]

refined_df = old_df[available_base_columns].copy()
print("Base columns retained:", available_base_columns)
print("Columns removed from refined dataset:", [col for col in removed_columns if col in old_df.columns])


Base columns retained: ['student_id', 'Curricular units 1st sem (approved)', 'Curricular units 1st sem (grade)', 'Curricular units 1st sem (enrolled)', 'Curricular units 1st sem (evaluations)', 'Admission grade']
Columns removed from refined dataset: ['quiz_average', 'late_submission_count', 'lms_resource_access', 'assignment_average', 'lms_login_frequency', 'ct1_score', 'ct2_score']


In [4]:
def min_max_scale(series):
    values = series.astype(float)
    min_value = values.min()
    max_value = values.max()
    if max_value == min_value:
        return pd.Series(np.full(len(values), 0.5), index=series.index)
    return (values - min_value) / (max_value - min_value)

n_students = len(old_df)

approved = old_df.get("Curricular units 1st sem (approved)", pd.Series(np.zeros(n_students)))
grade = old_df.get("Curricular units 1st sem (grade)", pd.Series(np.zeros(n_students)))
admission = old_df.get("Admission grade", pd.Series(np.full(n_students, 125)))

ability = (
    0.45 * min_max_scale(grade).to_numpy()
    + 0.35 * min_max_scale(approved).to_numpy()
    + 0.20 * min_max_scale(admission).to_numpy()
)
ability = np.clip(ability + rng.normal(0, 0.09, n_students), 0, 1)

semesters = rng.choice(
    [1, 2, 3, 4, 5, 6, 7, 8],
    size=n_students,
    p=[0.16, 0.16, 0.15, 0.14, 0.13, 0.11, 0.08, 0.07],
)

attendance = 52 + 43 * ability + rng.normal(0, 9, n_students)
attendance = np.clip(attendance, 35, 100).round(1)

ca1_score = 4 + 25 * ability + rng.normal(0, 4.2, n_students)
ca1_score = np.clip(ca1_score, 0, 30)

ca2_expected = np.select(
    [ca1_score >= 27, ca1_score >= 20, ca1_score >= 10],
    [24.5, 21.0, 14.0],
    default=8.5,
)
ca2_noise = rng.normal(0, 3.6, n_students) + rng.normal(0, 2.0, n_students) * (1 - ability)
ca2_score = 0.58 * ca1_score + 0.42 * ca2_expected + ca2_noise
ca2_score = np.clip(ca2_score, 0, 30)

ca3_score = 5 + 17 * ability + 0.18 * ca1_score + 0.14 * ca2_score + rng.normal(0, 4.6, n_students)
ca3_score = np.clip(ca3_score, 0, 30)

ca1_score = np.round(ca1_score, 1)
ca2_score = np.round(ca2_score, 1)
ca3_score = np.round(ca3_score, 1)

ca_scores = np.column_stack([ca1_score, ca2_score, ca3_score])
best_2_ca_average = np.sort(ca_scores, axis=1)[:, -2:].mean(axis=1)

mid_term_score = (
    0.38 * ca1_score
    + 0.42 * ca2_score
    + 4.5 * ability
    + rng.normal(0, 3.7, n_students)
)
mid_term_score = np.clip(mid_term_score, 0, 30)

previous_tgpa = 3.2 + 6.4 * ability + rng.normal(0, 0.9, n_students)
previous_tgpa = np.clip(previous_tgpa, 0, 10).round(2)
previous_tgpa = previous_tgpa.astype(float)
previous_tgpa[semesters == 1] = np.nan

current_signal = (best_2_ca_average / 30) * 0.45 + (mid_term_score / 30) * 0.35 + (attendance / 100) * 0.20
previous_signal = np.nan_to_num(previous_tgpa / 10, nan=ability)
trend_delta = current_signal - previous_signal + rng.normal(0, 0.10, n_students)
academic_trend = np.select(
    [trend_delta > 0.09, trend_delta < -0.09],
    ["Improving", "Declining"],
    default="Stable",
)

trend_risk = pd.Series(academic_trend).map({
    "Improving": -0.45,
    "Stable": 0.0,
    "Declining": 0.55,
}).to_numpy()

risk_score = (
    1.15 * (1 - attendance / 100)
    + 1.25 * (1 - best_2_ca_average / 30)
    + 1.05 * (1 - mid_term_score / 30)
    + 0.75 * (1 - np.nan_to_num(previous_tgpa / 10, nan=ability))
    + trend_risk
    + rng.normal(0, 0.42, n_students)
)

risk_probability = 1 / (1 + np.exp(-(risk_score - 2.35)))
academic_risk = rng.binomial(1, np.clip(risk_probability, 0.03, 0.97)).astype(float)

refined_df["semester"] = semesters
refined_df["attendance_percentage"] = attendance
refined_df["ca1_score"] = ca1_score
refined_df["ca2_score"] = ca2_score
refined_df["ca3_score"] = ca3_score
refined_df["best_2_ca_average"] = best_2_ca_average.round(1)
refined_df["mid_term_score"] = mid_term_score.round(1)
refined_df["previous_semester_tgpa"] = previous_tgpa
refined_df["academic_trend"] = academic_trend
refined_df[target_column] = academic_risk

refined_df.to_csv(REFINED_DATA_PATH, index=False)
print(f"Saved refined dataset to {REFINED_DATA_PATH}")
print("Refined shape:", refined_df.shape)


Saved refined dataset to ../data/ascentra_student_data_refined.csv
Refined shape: (4424, 16)


In [8]:
main_numeric_features = [
    "attendance_percentage",
    "ca1_score",
    "ca2_score",
    "ca3_score",
    "best_2_ca_average",
    "mid_term_score",
    "previous_semester_tgpa",
]

recomputed_best_2 = np.sort(
    refined_df[["ca1_score", "ca2_score", "ca3_score"]].to_numpy(),
    axis=1,
)[:, -2:].mean(axis=1).round(1)

best_2_matches = np.allclose(
    refined_df["best_2_ca_average"].to_numpy(),
    recomputed_best_2,
)

# print("Dataset shape:", refined_df.shape)
# print("Column names:")
# print(refined_df.columns.tolist())
# print("\nMissing-value counts:")
# print(refined_df.isna().sum())
# print("\nDescriptive statistics:")
# display(refined_df.describe(include="all"))
# print("\nValue ranges for CA and mid-term:")
# print(refined_df[["ca1_score", "ca2_score", "ca3_score", "mid_term_score"]].agg(["min", "max"]))
# print("\nSemester distribution:")
# print(refined_df["semester"].value_counts().sort_index())
# print("\nMissing previous TGPA values:", refined_df["previous_semester_tgpa"].isna().sum())
# print("Semester 1 rows:", (refined_df["semester"] == 1).sum())
# print("Later-semester missing TGPA values:", refined_df.loc[refined_df["semester"] > 1, "previous_semester_tgpa"].isna().sum())
# print("\nCorrelation matrix:")
# display(refined_df[main_numeric_features].corr())
# print("\nClass distribution:")
# print(refined_df[target_column].value_counts(normalize=False).sort_index())
# print(refined_df[target_column].value_counts(normalize=True).sort_index().round(3))
# print("\nBest-2 CA average correct:", best_2_matches)

# corr = refined_df[main_numeric_features].corr()
# print("CA1-CA2 correlation:", round(corr.loc["ca1_score", "ca2_score"], 3))
# print("Maximum CA-pair correlation:", round(corr.loc[["ca1_score", "ca2_score", "ca3_score"], ["ca1_score", "ca2_score", "ca3_score"]].where(~np.eye(3, dtype=bool)).max().max(), 3))
# print("Mid-term correlation with CA1:", round(corr.loc["mid_term_score", "ca1_score"], 3))
# print("Mid-term correlation with CA2:", round(corr.loc["mid_term_score", "ca2_score"], 3))
# print("Deleted features remaining:", sorted(set(removed_columns).intersection(refined_df.columns)))
refined_df.head(20)


,student_id,Curricular units 1st sem (approved),Curricular units 1st sem (grade),Curricular units 1st sem (enrolled),Curricular units 1st sem (evaluations),Admission grade,semester,attendance_percentage,ca1_score,ca2_score,ca3_score,best_2_ca_average,mid_term_score,previous_semester_tgpa,academic_trend,academic_risk
0,STU0001,0,0.000000,0,0,127.3,2,42.4,5.3,8.9,5.1,7.1,11.3,3.97,Stable,1.0
1,STU0002,6,14.000000,6,6,142.5,4,58.4,15.1,15.9,16.8,16.4,9.6,6.08,Stable,0.0
2,STU0003,0,0.000000,6,0,124.8,6,50.6,6.8,8.8,3.4,7.8,5.0,3.96,Stable,0.0
3,STU0004,6,13.428571,6,8,119.6,2,55.9,24.0,16.3,15.2,20.2,12.5,6.17,Stable,0.0
4,STU0005,5,12.333333,6,9,141.5,5,76.9,15.3,13.2,10.6,14.2,19.4,4.27,Improving,0.0
5,STU0006,5,11.857143,5,10,114.8,4,67.1,12.1,16.4,8.5,14.2,15.9,5.87,Stable,1.0
6,STU0007,7,13.300000,7,9,128.4,4,72.9,13.1,12.5,14.6,13.8,5.0,6.57,Declining,0.0
7,STU0008,0,0.000000,5,5,113.1,1,35.0,0.8,7.8,14.4,11.1,3.4,NaN,Stable,1.0
8,STU0009,6,13.875000,6,8,129.3,5,56.3,24.3,24.0,20.8,24.2,20.4,6.78,Stable,0.0
9,STU0010,5,11.400000,6,9,123.0,7,65.8,7.8,6.8,16.4,12.1,4.5,6.52,Declining,0.0
